In [125]:
from feat import Detector
import numpy as np
import os
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image
import joblib
from time import time
from tqdm import tqdm
from glob import glob

In [135]:
imgfolder = Path('/media/landry/fastscratch/landry_dev_scratch/test_outputs/vframes')
imgs = list(imgfolder.glob('*.png'))
detector = Detector(device='cuda', n_jobs=16)

In [51]:
pose_data = joblib.load('/media/landry/fastscratch/landry_dev_scratch/test_outputs/cam2_concatenated_trimmed_trimmed_test.pkl')

In [8]:
# need to format the detected_faces argument so that its a
# list of lists with the same length as the number of frames. 
# Each list item is a list containing the (x1, y1, x2, y2) coordinates of each detected face in that frame.
# So I could just make those coordinates based on the size of the image, and then just make a list of those lists

In [53]:
# get the shape of the first image
img = imgs[0]
img = Image.open(img)
img = np.array(img)
h, w, _ = img.shape
print(h, w)
# look at the image
px.imshow(img)

1080 1920


In [140]:
def get_face(joints2d_frame):
    head = joints2d_frame[15]
    face_bbox = [head[0]-75, head[1]+75, head[0]+75, head[1]-75]
    face_bbox = [int(round(i)) for i in face_bbox]
    # return as x1, y1, x2, y2
    face_bbox = [face_bbox[0], face_bbox[3], face_bbox[2], face_bbox[1]]
    return face_bbox

def add_faces(pose_data):
    for person, data in pose_data.items():
        pose_data[person]['face'] = {}
        bboxes = []
        for frame, joints2d in enumerate(data['smpl_joints2d']):
            bboxes.append(get_face(joints2d))
        pose_data[person]['face']['bbox'] = np.array(bboxes)

add_faces(pose_data)

In [141]:
# view face for frame 0
img = imgs[0]
img = Image.open(img)
img = np.array(img)
face_bbox = pose_data[0]['face']['bbox'][0]
face = img[face_bbox[1]:face_bbox[3], face_bbox[0]:face_bbox[2]]

px.imshow(face)

In [151]:
def get_face_img(pose_data, track, frame):
    # this will return a np array representing the face in the frame
    # first check if frame is in frame_ids for that track
    if frame not in pose_data[track]['frame_ids']:
        raise ValueError('frame not in frame_ids for this track')
    else:
        loc = np.where(pose_data[track]['frame_ids']==frame)[0][0]
        face_bbox = pose_data[track]['face']['bbox'][loc]
        img = imgs[frame]
        img = Image.open(img)
        img = np.array(img)
        face = img[face_bbox[1]:face_bbox[3], face_bbox[0]:face_bbox[2]]
        return face

In [152]:
tface = get_face_img(pose_data, 1, 1200)
px.imshow(tface)

In [63]:
fbbox = pose_data[0]['face']['bbox'][0].tolist()
fbbox.append(0) 
fbbox

[422, 115, 622, 315, 0]

In [64]:
tbbox = detector.detect_faces(tface)
# this is x1, y1, x2, y2, confidence
tbbox

[[[101.95088195800781,
   62.06728744506836,
   180.66575622558594,
   167.96627807617188,
   0.9932805]]]

In [104]:
tlandmarks = detector.detect_landmarks(tface, tbbox)[0][0]
# plot the landmarks on tface
fig = px.imshow(tface)
fig.add_trace(go.Scatter(x=tlandmarks[:,0], y=tlandmarks[:,1], mode='markers'))

In [95]:
# add landmarks to pose_data
def add_landmarks(pose_data, detector, images):
    fcount = 0
    pdat = pose_data.copy()
    for person, data in pose_data.items():
        pose_data[person]['face']['landmarks'] = {}
        frames = data['frame_ids']
        landmarks = []
        for frame in tqdm(frames):
            fcount += 1
            face = get_face(pose_data, person, frame)
            fbbox = pose_data[person]['face']['bbox'][frame].tolist()
            fbbox.append(1.0)
            tbbox = detector.detect_faces(face)
            tlandmarks = detector.detect_landmarks(face, tbbox)[0][0]
            landmarks.append(tlandmarks)

        pdat[person]['face']['landmarks'] = np.array(landmarks)
    return pdat, fcount

ltime = time()
pdata, fc = add_landmarks(pose_data, detector, imgs)
print(f'landmarks took {time()-ltime} seconds for {len(pose_data)} people for {fc} frames')

100%|██████████| 1619/1619 [01:23<00:00, 19.39it/s]

landmarks took 166.57411122322083 seconds for 2 people for 3238 frames


In [116]:
# do the same thing for AUs
def get_au(pdata, track, frame):
    # see if frame is in frame_ids for that track
    if frame not in pdata[track]['frame_ids']:
        raise ValueError('frame not in frame_ids for this track')
    else:
        loc = np.where(pdata[track]['frame_ids']==frame)[0][0]

    face_img = get_face(pdata, track, frame)
    landmarks = [[pdata[track]['face']['landmarks'][loc]]]
    return detector.detect_aus(face_img, landmarks)

au = get_au(pdata, 1, 600)    
len(au[0][0])

20

In [122]:
pdata[0]['face']['landmarks'].shape

(1619, 68, 2)

In [123]:
def add_aus(pose_data, detector, images):
    fcount = 0
    pdat = pose_data.copy()
    for person, data in pose_data.items():
        pose_data[person]['face']['aus'] = {}
        frames = data['frame_ids']
        aus = []
        for frame in tqdm(frames):
            fcount += 1
            aus.append(get_au(pose_data, person, frame))

        pdat[person]['face']['aus'] = np.array(aus)
    return pdat, fcount

atime = time()
pdata, fc = add_aus(pdata, detector, imgs)
print(f'AUs took {time()-atime} seconds for {len(pose_data)} people for {fc} frames')

  5%|▍         | 77/1619 [00:13<04:36,  5.57it/s]


KeyboardInterrupt: 

In [127]:
faceimgs = glob('/media/landry/fastscratch/landry_dev_scratch/test_outputs/face_images/*.png')
faceimgs.sort()

In [136]:
fexdf = detector.detect_image(faceimgs, batch_size=64)

100%|██████████| 26/26 [03:08<00:00,  7.26s/it]


In [139]:
display(fexdf[[i for i in fexdf.columns if 'Rect' in i]])

,FaceRectX,FaceRectY,FaceRectWidth,FaceRectHeight
0,62.855442,47.346161,77.326504,108.225464
1,62.855442,47.346161,77.326504,108.225464
2,62.855442,47.346161,77.326504,108.225464
3,62.855442,47.346161,77.326504,108.225464
4,62.855442,47.346161,77.326504,108.225464
...,...,...,...,...
1613,63.936703,49.694641,79.990467,106.331177
1614,65.052917,49.261822,79.260757,106.690006
1615,63.645576,46.930397,79.388573,107.815331
1616,62.986088,48.810490,80.239223,107.143566


In [161]:
#### Starting over so I can write an annotation scirpt for faces given pose data ####
def get_face(joints2d_frame):
    head = joints2d_frame[15]
    face_bbox = [head[0]-75, head[1]+75, head[0]+75, head[1]-75]
    face_bbox = [int(round(i)) for i in face_bbox]
    # return as x1, y1, x2, y2
    face_bbox = [face_bbox[0], face_bbox[3], face_bbox[2], face_bbox[1]]
    return face_bbox

def add_faces(pose_data):
    for person, data in pose_data.items():
        pose_data[person]['face'] = {}
        bboxes = []
        for frame, joints2d in enumerate(data['smpl_joints2d']):
            bboxes.append(get_face(joints2d))
        pose_data[person]['face']['bbox'] = np.array(bboxes)

def get_face_img(pose_data, track, frame, imgs):
    """
    This function will return a np array representing the face in the frame
    @param pose_data: dict, the pose data
    @param track: int, the track number
    @param frame: int, the frame number
    @param imgs: list, list of image paths
    """

    # this will return a np array representing the face in the frame
    # first check if frame is in frame_ids for that track
    if frame not in pose_data[track]['frame_ids']:
        raise ValueError('frame not in frame_ids for this track')
    else:
        loc = np.where(pose_data[track]['frame_ids']==frame)[0][0]
        face_bbox = pose_data[track]['face']['bbox'][loc]
        img = np.array(Image.open(imgs[frame]))
        face = img[face_bbox[1]:face_bbox[3], face_bbox[0]:face_bbox[2]]
        return face
    
import concurrent.futures

def extract_face_images(pose_data, data_dir, image_dir):
    face_folder = Path(data_dir) / 'face_images'
    if not os.path.exists(face_folder):
        os.mkdir(face_folder)
    image_paths = list(Path(image_dir).glob('*.png'))
    image_paths.sort()

    def save_face(person, frame):
        face = get_face_img(pose_data, person, frame, image_paths)
        face_path = f'{face_folder}/track_{person}/frame_{frame}.png'
        Image.fromarray(face).save(face_path)

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for person, data in pose_data.items():
            if not os.path.exists(f'{face_folder}/track_{person}'):
                os.mkdir(f'{face_folder}/track_{person}')
            for frame in tqdm(data['frame_ids']):
                futures.append(executor.submit(save_face, person, frame))
        
        # Wait for all the futures to complete
        concurrent.futures.wait(futures)


In [163]:
start = time()
extract_face_images(pose_data, '/media/landry/fastscratch/landry_dev_scratch/test_outputs', '/media/landry/fastscratch/landry_dev_scratch/test_outputs/vframes')
print(f'face extraction took {time()-start} seconds')

  0%|          | 0/1619 [00:00<?, ?it/s]

100%|██████████| 1619/1619 [00:00<00:00, 109143.45it/s]


face extraction took 66.46453189849854 seconds
